In [10]:
# Uncomment if running in a fresh environment missing any packages:
# !pip install -q transformers torch pandas pyarrow nltk seaborn matplotlib scikit-learn tqdm

import re
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import nltk
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from nltk.tokenize import sent_tokenize
from sklearn.metrics import classification_report
from tqdm import tqdm  # Standard console tqdm avoids Colab widget bloat on GitHub
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers.utils import logging as hf_logging

# Suppress verbose Hugging Face download/loading warnings for cleaner notebook output
hf_logging.set_verbosity_error()

# Ensure NLTK sentence tokenizer data is available
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# Smart Path Resolution: Automatically detects Colab (/content) vs. Local Repo (data/)
if Path("/content/competitor_app_reviews.parquet").exists():
    DATA_DIR = Path("/content")
    RAW_DATA_PATH = DATA_DIR / "competitor_app_reviews.parquet"
    OUTPUT_DIR = DATA_DIR / "outputs"
else:
    DATA_DIR = Path("data")
    RAW_DATA_PATH = DATA_DIR / "raw" / "competitor_app_reviews.parquet"
    OUTPUT_DIR = DATA_DIR / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLAUSES_PATH = OUTPUT_DIR / "cleaned_review_clauses.parquet"
RESULTS_PATH = OUTPUT_DIR / "absa_stakeholder_results.parquet"
SUMMARY_CSV_PATH = OUTPUT_DIR / "stakeholder_aspect_comparison.csv"
CHART_PNG_PATH = OUTPUT_DIR / "nss_competitive_comparison.png"

print(f"Environment ready. Reading from: {RAW_DATA_PATH} | Saving outputs to: {OUTPUT_DIR}")

Environment ready. Reading from: /content/competitor_app_reviews.parquet | Saving outputs to: /content/outputs


In [13]:
def load_and_clean_parquet(file_path: Path | str = RAW_DATA_PATH) -> pd.DataFrame:
    """
    Loads the Play Store parquet dataset, safely standardises schema columns,
    normalises app identifiers, and filters out scraper spam/noise.
    """
    df = pd.read_parquet(file_path)
    print(f"Loaded raw dataset with shape: {df.shape}")

    # Ordered priority mapping prevents duplicate target column collisions
    schema_priority = {
        "review_text": ["review_text", "content", "review", "text", "body"],
        "app_name": ["app_name", "app", "title", "app_id", "package_name"],
        "rating": ["rating", "score", "stars", "star"],
        "review_date": ["review_date", "at", "created_at", "timestamp", "date"],
        "review_id": ["review_id", "reviewid", "id"],
    }

    col_mapping = {}
    lower_to_orig = {col.lower(): col for col in df.columns}

    for target_col, candidates in schema_priority.items():
        if target_col in df.columns:
            continue  # Target column already exists cleanly
        for candidate in candidates:
            if candidate in lower_to_orig:
                col_mapping[lower_to_orig[candidate]] = target_col
                break

    df = df.rename(columns=col_mapping)

    # Normalise app names
    if "app_name" in df.columns:
        df["app_name"] = (
            df["app_name"]
            .astype(str)
            .apply(
                lambda x: "Spotify"
                if "spotify" in x.lower()
                else ("YouTube Music" if "youtube" in x.lower() else x)
            )
        )

    # Drop missing text and strip whitespace
    df = df.dropna(subset=["review_text"]).copy()
    df["review_text"] = df["review_text"].astype(str).str.strip()

    def is_valid_review(text: str) -> bool:
        if len(text) < 5:
            return False
        # Filter known scraper weather copy-paste spam
        if "FIRST BIG COLD FRONT THIS WEEKEND" in text.upper():
            return False
        # Filter symbol/emoji-only spam or keyboard smashes
        alpha_ratio = sum(c.isalpha() for c in text) / len(text)
        return alpha_ratio > 0.4

    df = df[df["review_text"].apply(is_valid_review)].reset_index(drop=True)
    print(f"Cleaned dataset shape: {df.shape}")
    return df


def split_reviews_into_clauses(df: pd.DataFrame) -> pd.DataFrame:
    """
    Splits multi-sentence and contrastive reviews ('Great sound, but terrible ads')
    into granular clauses so opposing sentiments are evaluated independently.
    """
    records = []
    contrastive_pattern = re.compile(
        r"\b(?:but|however|although|though|except|yet)\b", re.IGNORECASE
    )

    for idx, row in df.iterrows():
        sentences = sent_tokenize(row["review_text"])
        for sent in sentences:
            sub_clauses = contrastive_pattern.split(sent)
            for clause in sub_clauses:
                clause = clause.strip(" ,;:-")
                if len(clause) >= 4:
                    records.append(
                        {
                            "review_id": row.get("review_id", idx),
                            "app_name": row.get("app_name", "Unknown"),
                            "rating": row.get("rating"),
                            "review_date": row.get("review_date"),
                            "full_review": row["review_text"],
                            "clause": clause,
                        }
                    )

    clause_df = pd.DataFrame(records)
    print(f"Extracted {len(clause_df):,} aspect-ready clauses from {len(df):,} reviews.")
    return clause_df


df_clean = load_and_clean_parquet()
df_clauses = split_reviews_into_clauses(df_clean)
df_clauses.to_parquet(CLAUSES_PATH, index=False)

# Quick sanity check (5 rows instead of 100)
df_clean[["app_name", "rating", "review_text"]].head(5)

Loaded raw dataset with shape: (11093, 11)
Cleaned dataset shape: (11071, 11)
Extracted 24,261 aspect-ready clauses from 11,071 reviews.


,app_name,rating,review_text
0,Spotify,5,Good and clean UI Better with a premium subscr...
1,Spotify,5,I recently installed this app on my phone but ...
2,Spotify,3,Doesn't recognise USB DAC on Android like Powe...
3,Spotify,1,so 30 mins of ad-free listening doesn't even e...
4,Spotify,5,amazing experience since day 1


In [14]:
# Stakeholder Aspect Taxonomy (Includes English + Hinglish/Spanish/Indonesian terms)
ASPECT_TAXONOMY: Dict[str, List[str]] = {
    "Monetization, Ads & Free Tier": [
        r"\bads?\b", r"\badvertis\w*", r"\bpremium\b", r"\bsubscri\w*", r"\bpay\w*",
        r"\bprice\b", r"\bmoney\b", r"\bfree\s*tier\b", r"\bskips?\b", r"\bbackground\s*play\b",
        r"\bgreedy\b", r"\biklan\b", r"\banuncios?\b", r"\breclame\b", r"\bpaisa\b",
    ],
    "Shuffle, Queue & Recommendations": [
        r"\bshuffl\w*", r"\bsmart\s*shuffle\b", r"\bqueue\w*", r"\brecommend\w*",
        r"\balgorithm\w*", r"\brepeat\w*", r"\bloop\w*", r"\bai\s*dj\b", r"\bautoplay\b",
        r"\bmix(?:es)?\b", r"\bdiscover\s*weekly\b", r"\bdaylist\b",
    ],
    "Audio Quality & Playback": [
        r"\baudio\b", r"\bsound\s*quality\b", r"\blossless\b", r"\bdolby\b", r"\batmos\b",
        r"\bequalizer\b", r"\bcrossfade\b", r"\bvolume\b", r"\bbit\s*rate\b", r"\b320\s*kbps\b",
        r"\bflac\b", r"\bbass\b", r"\bgapless\b",
    ],
    "Catalog, Podcasts & Content": [
        r"\bpodcast\w*", r"\baudiobook\w*", r"\bcatalog\w*", r"\blibrary\b", r"\blyrics?\b",
        r"\bremix\w*", r"\bcovers?\b", r"\bexplicit\b", r"\bcensor\w*", r"\bai\s*slop\b",
        r"\bai\s*generat\w*", r"\bsongs?\s*missing\b", r"\bepisode\w*",
    ],
    "UI/UX, Widgets & Organization": [
        r"\bui\b", r"\bux\b", r"\binterface\b", r"\bwidget\w*", r"\b4x1\b", r"\b4x2\b",
        r"\bsort\w*", r"\bfolder\w*", r"\bsearch\b", r"\blayout\b", r"\bdesign\b",
        r"\bsound\s*capsule\b", r"\bplaylist\w*", r"\bnavigat\w*",
    ],
    "App Stability, Offline & Storage": [
        r"\boffline\b", r"\bdownload\w*", r"\bcache\b", r"\bstorage\b", r"\bcrash\w*",
        r"\blag\w*", r"\bbug\w*", r"\bfreez\w*", r"\bglitch\w*", r"\bbattery\b",
        r"\bandroid\s*auto\b", r"\bbuffer\w*", r"\bwifi\b", r"\binternet\b",
    ],
}

COMPILED_TAXONOMY = {
    category: re.compile("|".join(patterns), re.IGNORECASE)
    for category, patterns in ASPECT_TAXONOMY.items()
}


def detect_aspects(clause: str) -> List[Tuple[str, str]]:
    """
    Identifies stakeholder aspect categories AND extracts the exact matched term
    so DeBERTa evaluates sentiment against a natural, concrete aspect term.
    """
    matches = []
    for category, pattern in COMPILED_TAXONOMY.items():
        match = pattern.search(clause)
        if match:
            matches.append((category, match.group(0).lower()))
    return matches


class DeBERTaABSAEngine:
    """
    Runs Aspect-Based Sentiment Analysis using a fine-tuned DeBERTa-v3 model
    trained on (Sentence, Aspect Term) pairs.
    """

    def __init__(
        self,
        model_name: str = "yangheng/deberta-v3-base-absa-v1.1",
        batch_size: int = 64,
    ):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Initializing {model_name} on {self.device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(
            self.device
        )
        self.model.eval()
        self.batch_size = batch_size
        self.id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}

    def predict_batch(self, texts: List[str], aspect_terms: List[str]) -> List[dict]:
        results = []
        for i in tqdm(
            range(0, len(texts), self.batch_size), desc="Running ABSA Inference"
        ):
            batch_texts = texts[i : i + self.batch_size]
            batch_aspects = aspect_terms[i : i + self.batch_size]

            encoded = self.tokenizer(
                batch_texts,
                batch_aspects,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt",
            ).to(self.device)

            with torch.inference_mode():
                outputs = self.model(**encoded)
                probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()

            for prob in probs:
                pred_idx = int(prob.argmax())
                sentiment_score = float(prob[2] - prob[0])  # Range: [-1.0, +1.0]
                results.append(
                    {
                        "sentiment": self.id2label[pred_idx],
                        "confidence": round(float(prob[pred_idx]), 4),
                        "sentiment_score": round(sentiment_score, 4),
                    }
                )
        return results


def run_absa_pipeline(clause_path: Path | str = CLAUSES_PATH) -> pd.DataFrame:
    df_clauses = pd.read_parquet(clause_path)

    # Detect (category, exact_aspect_term) tuples per clause
    df_clauses["aspect_matches"] = df_clauses["clause"].apply(detect_aspects)

    df_exploded = (
        df_clauses[df_clauses["aspect_matches"].map(len) > 0]
        .explode("aspect_matches")
        .reset_index(drop=True)
    )

    df_exploded[["aspect_category", "aspect_term"]] = pd.DataFrame(
        df_exploded["aspect_matches"].tolist(), index=df_exploded.index
    )
    df_exploded = df_exploded.drop(columns=["aspect_matches"])

    print(f"Total (Clause, Aspect) pairs to evaluate: {len(df_exploded):,}")

    # Pass the concrete matched term (e.g., 'ads') rather than the broad bucket name
    engine = DeBERTaABSAEngine(batch_size=64)
    predictions = engine.predict_batch(
        texts=df_exploded["clause"].tolist(),
        aspect_terms=df_exploded["aspect_term"].tolist(),
    )

    final_df = pd.concat([df_exploded, pd.DataFrame(predictions)], axis=1)
    final_df.to_parquet(RESULTS_PATH, index=False)
    print(f"Saved ABSA results to '{RESULTS_PATH}'")
    return final_df


df_results = run_absa_pipeline()

Total (Clause, Aspect) pairs to evaluate: 11,328
Initializing yangheng/deberta-v3-base-absa-v1.1 on cuda...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Running ABSA Inference: 100%|██████████| 177/177 [00:56<00:00,  3.13it/s]


Saved ABSA results to '/content/outputs/absa_stakeholder_results.parquet'


In [19]:
def generate_stakeholder_summary(results_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregates aspect-level sentiment metrics into a competitive comparison table."""
    summary = (
        results_df.groupby(["app_name", "aspect_category"])
        .agg(
            total_mentions=("sentiment", "count"),
            positive_count=("sentiment", lambda s: (s == "Positive").sum()),
            neutral_count=("sentiment", lambda s: (s == "Neutral").sum()),
            negative_count=("sentiment", lambda s: (s == "Negative").sum()),
            avg_model_score=("sentiment_score", "mean"),
        )
        .reset_index()
    )

    summary["pct_positive"] = (
        summary["positive_count"] / summary["total_mentions"] * 100
    ).round(1)
    summary["pct_neutral"] = (
        summary["neutral_count"] / summary["total_mentions"] * 100
    ).round(1)
    summary["pct_negative"] = (
        summary["negative_count"] / summary["total_mentions"] * 100
    ).round(1)

    # Net Sentiment Score (NSS) = % Positive - % Negative (scales from -100 to +100)
    summary["net_sentiment_score"] = (
        summary["pct_positive"] - summary["pct_negative"]
    ).round(1)

    summary.to_csv(SUMMARY_CSV_PATH, index=False)
    return summary
df_summary = generate_stakeholder_summary(df_results)
df_summary.head()



,app_name,aspect_category,total_mentions,positive_count,neutral_count,negative_count,avg_model_score,pct_positive,pct_neutral,pct_negative,net_sentiment_score
0,Spotify,"App Stability, Offline & Storage",648,122,119,407,-0.421052,18.8,18.4,62.8,-44.0
1,Spotify,Audio Quality & Playback,191,93,28,70,0.119099,48.7,14.7,36.6,12.1
2,Spotify,"Catalog, Podcasts & Content",478,120,226,132,-0.013957,25.1,47.3,27.6,-2.5
3,Spotify,"Monetization, Ads & Free Tier",2862,463,634,1765,-0.437262,16.2,22.2,61.7,-45.5
4,Spotify,"Shuffle, Queue & Recommendations",674,171,111,392,-0.321096,25.4,16.5,58.2,-32.8
